**Análise de Qualidade e Eficiência no Atendimento ao Consumidor**

versão 1 - 01mar26

Análise e agregações

In [ ]:
import pandas as pd
import seaborn as srn
import numpy as np
import statistics as sts
import matplotlib.pyplot as plt
import duckdb

In [ ]:
# Specify the path to your Parquet file in the Colab environment
parquet_file_path = '/content/drive/MyDrive/Colab Notebooks/2026/arquivos/base_completa_2025_12.parquet'

# Ensure the DuckDB connection 'con' is available. If not, re-establish it.
if 'con' not in locals():
    import duckdb
    con = duckdb.connect(database=':memory:', read_only=False)
    print("DuckDB connection re-established.")

table_name = "base_completa_2025_12"

try:
    # Use DuckDB's read_parquet to load the Parquet file
    con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM read_parquet('{parquet_file_path}')")
    print(f"Successfully loaded '{parquet_file_path}' into DuckDB table '{table_name}'.")

    # Display the first 3 rows of the newly created table
    print(f"First 3 rows of '{table_name}':")
    display(con.execute(f"SELECT * FROM {table_name} LIMIT 3").fetchdf())

except Exception as e:
    print(f"Error loading {parquet_file_path} into DuckDB: {e}")

DuckDB connection re-established.
Successfully loaded '/content/drive/MyDrive/Colab Notebooks/2026/arquivos/base_completa_2025_12.parquet' into DuckDB table 'base_completa_2025_12'.
First 3 rows of 'base_completa_2025_12':


,Gestor,Região,UF,Cidade,Sexo,Faixa Etária,Ano Abertura,Mês Abertura,Data Abertura,Data Resposta,...,Assunto,Grupo Problema,Problema,Como Comprou Contratou,Procurou Empresa,Respondida,Situação,Avaliação Reclamação,Nota do Consumidor,Análise da Recusa
0,Programa Estadual de Proteção e Defesa do Cons...,SE,MG,Barroso,M,entre 41 a 50 anos,2025,10,2025-10-10,NaT,...,Crédito Pessoal e Demais Empréstimos (exceto f...,Cobrança / Contestação,Negativação indevida - desconhece motivo e/ou ...,Não comprei / contratei,N,N,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente
1,Fundação de Proteção e Defesa do Consumidor,SE,SP,São Paulo,F,entre 41 a 50 anos,2025,10,2025-10-11,2025-11-05,...,"Vestuário e Artigos de Uso Pessoal (roupa, cal...",Atendimento / SAC,Má qualidade no atendimento (descortesia / des...,Ganhei de presente,S,S,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente
2,Secretaria Nacional do Consumidor,S,RS,Sant'Ana do Livramento,O,entre 21 a 30 anos,2025,10,2025-10-12,NaT,...,"Produtos relacionados a saúde, exceto medicame...",Contrato / Oferta,Recusa em cancelar compra/serviço no prazo de ...,Internet,S,N,Finalizada não avaliada,Não Avaliada,<NA>,Improcedente


In [ ]:
print("Tables currently available in DuckDB:")
display(con.execute("SHOW TABLES").fetchdf())

Tables currently available in DuckDB:


,name
0,base_completa_2025_12


Índice de Resolução / Competitividade

In [ ]:
sql_query = """
SELECT
    "Segmento de Mercado",
    SUM(CASE WHEN "Situação" = 'Finalizada avaliada' AND "Avaliação Reclamação" = 'Resolvida' THEN 1 ELSE 0 END) AS "Reclamações Resolvidas",
    SUM(CASE WHEN "Situação" = 'Finalizada não avaliada' THEN 1 ELSE 0 END) AS "Reclamações Não Avaliadas",
    SUM(CASE WHEN "Situação" IN ('Finalizada avaliada', 'Finalizada não avaliada') THEN 1 ELSE 0 END) AS "Total Finalizadas",
    CAST(
        SUM(CASE WHEN "Situação" = 'Finalizada avaliada' AND "Avaliação Reclamação" = 'Resolvida' THEN 1 ELSE 0 END) +
        SUM(CASE WHEN "Situação" = 'Finalizada não avaliada' THEN 1 ELSE 0 END)
    AS DOUBLE) *
    100.0 /
    NULLIF(SUM(CASE WHEN "Situação" IN ('Finalizada avaliada', 'Finalizada não avaliada') THEN 1 ELSE 0 END), 0)
    AS "Índice de Solução"
FROM
    base_completa_2025_12
GROUP BY
    "Segmento de Mercado"
ORDER BY
    "Índice de Solução" DESC
"""

indice_solucao_segmento_sql = con.execute(sql_query).fetchdf()

print("Índice de Solução por Segmento de Mercado (via SQL DuckDB):")
display(indice_solucao_segmento_sql)


Índice de Solução por Segmento de Mercado (via SQL DuckDB):


,Segmento de Mercado,Reclamações Resolvidas,Reclamações Não Avaliadas,Total Finalizadas,Índice de Solução
0,Shopping Centers,1.0,2.0,3.0,100.000000
1,"Operadoras de Telecomunicações (Telefonia, Int...",5064.0,12287.0,19355.0,89.646086
2,Empresas de Recuperação de Crédito,312.0,6552.0,7702.0,89.119709
3,"Hospitais, Clínicas, Laboratórios e Outros Ser...",5.0,10.0,17.0,88.235294
4,Cartões de Descontos,20.0,198.0,248.0,87.903226
5,Bancos de Dados e Cadastros de Consumidores,1312.0,18991.0,23166.0,87.641371
6,"Bancos, Financeiras e Administradoras de Cartão",5747.0,115519.0,138754.0,87.396399
7,Empresas de Pagamento Eletrônico,970.0,9517.0,12107.0,86.619311
8,Entidades Sem Fins Lucrativos,1.0,37.0,44.0,86.363636
9,Supermercados,135.0,164.0,347.0,86.167147


In [ ]:
# Define o caminho para salvar o arquivo Parquet no Google Drive
output_parquet_path = '/content/drive/MyDrive/Colab Notebooks/2026/arquivos/indice_solucao_segmento.parquet'

# Salva o DataFrame em um arquivo Parquet
indice_solucao_segmento_sql.to_parquet(output_parquet_path, index=False)

print(f"DataFrame 'indice_solucao_segmento_sql' salvo com sucesso em: {output_parquet_path}")

DataFrame 'indice_solucao_segmento_sql' salvo com sucesso em: /content/drive/MyDrive/Colab Notebooks/2026/arquivos/indice_solucao_segmento.parquet


Qualidade por Canal de Vendas

In [ ]:
sql_query_avg_nota = """
SELECT
    "Segmento de Mercado",
    "Como Comprou Contratou",
    AVG("Nota do Consumidor") AS "Media Nota do Consumidor"
FROM
    base_completa_2025_12
GROUP BY
    "Segmento de Mercado",
    "Como Comprou Contratou"
ORDER BY
    "Segmento de Mercado",
    "Como Comprou Contratou"
"""

media_nota_consumidor_compra_df = con.execute(sql_query_avg_nota).fetchdf()

print("Média da Nota do Consumidor por Segmento de Mercado e Como Comprou Contratou (via SQL DuckDB):")
display(media_nota_consumidor_compra_df)

Média da Nota do Consumidor por Segmento de Mercado e Como Comprou Contratou (via SQL DuckDB):


,Segmento de Mercado,Como Comprou Contratou,Media Nota do Consumidor
0,Administradoras de Consórcios,Catálogo,1.000000
1,Administradoras de Consórcios,Domicílio,3.000000
2,Administradoras de Consórcios,Internet,2.086957
3,Administradoras de Consórcios,Loja física,1.884615
4,Administradoras de Consórcios,Não comprei / contratei,2.259259
...,...,...,...
331,"Viagens, Turismo e Hospedagem",Loja física,3.750000
332,"Viagens, Turismo e Hospedagem",Não comprei / contratei,2.695652
333,"Viagens, Turismo e Hospedagem",SMS / Mensagem de texto,NaN
334,"Viagens, Turismo e Hospedagem","Stand, feiras e eventos",3.000000


In [ ]:
# Define o caminho para salvar o arquivo Parquet
output_parquet_avg_nota_path = '/content/drive/MyDrive/Colab Notebooks/2026/arquivos/media_nota_consumidor_compra.parquet'

# Salva o DataFrame em um arquivo Parquet
media_nota_consumidor_compra_df.to_parquet(output_parquet_avg_nota_path, index=False)

print(f"DataFrame 'media_nota_consumidor_compra_df' salvo com sucesso em: {output_parquet_avg_nota_path}")

DataFrame 'media_nota_consumidor_compra_df' salvo com sucesso em: /content/drive/MyDrive/Colab Notebooks/2026/arquivos/media_nota_consumidor_compra.parquet


Taxa de Retenção do SAC

In [ ]:
sql_query_procurou_empresa = """
SELECT
    "Segmento de Mercado",
    "Procurou Empresa",
    SUM(CASE WHEN "Situação" = 'Finalizada avaliada' AND "Avaliação Reclamação" = 'Resolvida' THEN 1 ELSE 0 END) AS "Reclamações Resolvidas",
    SUM(CASE WHEN "Situação" = 'Finalizada não avaliada' THEN 1 ELSE 0 END) AS "Reclamações Não Avaliadas",
    SUM(CASE WHEN "Situação" IN ('Finalizada avaliada', 'Finalizada não avaliada') THEN 1 ELSE 0 END) AS "Total Finalizadas",
    CAST(
        SUM(CASE WHEN "Situação" = 'Finalizada avaliada' AND "Avaliação Reclamação" = 'Resolvida' THEN 1 ELSE 0 END) +
        SUM(CASE WHEN "Situação" = 'Finalizada não avaliada' THEN 1 ELSE 0 END)
    AS DOUBLE) *
    100.0 /
    NULLIF(SUM(CASE WHEN "Situação" IN ('Finalizada avaliada', 'Finalizada não avaliada') THEN 1 ELSE 0 END), 0)
    AS "Índice de Solução"
FROM
    base_completa_2025_12
GROUP BY
    "Segmento de Mercado",
    "Procurou Empresa"
ORDER BY
    "Segmento de Mercado",
    "Procurou Empresa",
    "Índice de Solução" DESC
"""

indice_solucao_procurou_empresa_df = con.execute(sql_query_procurou_empresa).fetchdf()

print("Índice de Solução por Segmento de Mercado e 'Procurou Empresa' (via SQL DuckDB):")
display(indice_solucao_procurou_empresa_df)


Índice de Solução por Segmento de Mercado e 'Procurou Empresa' (via SQL DuckDB):


,Segmento de Mercado,Procurou Empresa,Reclamações Resolvidas,Reclamações Não Avaliadas,Total Finalizadas,Índice de Solução
0,Administradoras de Consórcios,N,3.0,44.0,57.0,82.456140
1,Administradoras de Consórcios,S,47.0,465.0,641.0,79.875195
2,Agua e Saneamento,N,15.0,133.0,179.0,82.681564
3,Agua e Saneamento,S,168.0,1257.0,1790.0,79.608939
4,Aluguel de Carros,N,10.0,41.0,57.0,89.473684
...,...,...,...,...,...,...
82,Varejo,S,218.0,1163.0,1764.0,78.287982
83,"Vestuário, Calçados e Acessórios",N,18.0,151.0,200.0,84.500000
84,"Vestuário, Calçados e Acessórios",S,272.0,850.0,1450.0,77.379310
85,"Viagens, Turismo e Hospedagem",N,10.0,106.0,143.0,81.118881


In [ ]:
# Define o caminho para salvar o arquivo Parquet
output_parquet_procurou_empresa_path = '/content/drive/MyDrive/Colab Notebooks/2026/arquivos/indice_solucao_procurou_empresa.parquet'

# Salva o DataFrame em um arquivo Parquet
indice_solucao_procurou_empresa_df.to_parquet(output_parquet_procurou_empresa_path, index=False)

print(f"DataFrame 'indice_solucao_procurou_empresa_df' salvo com sucesso em: {output_parquet_procurou_empresa_path}")

DataFrame 'indice_solucao_procurou_empresa_df' salvo com sucesso em: /content/drive/MyDrive/Colab Notebooks/2026/arquivos/indice_solucao_procurou_empresa.parquet


Tempo de Resposta

In [ ]:
sql_query_avg_tempo_resposta = """
SELECT
    "Segmento de Mercado",
    AVG("Tempo Resposta") AS "Media Tempo Resposta"
FROM
    base_completa_2025_12
GROUP BY
    "Segmento de Mercado"
ORDER BY
    "Media Tempo Resposta" DESC
"""

media_tempo_resposta_df = con.execute(sql_query_avg_tempo_resposta).fetchdf()

print("Média do Tempo de Resposta por Segmento de Mercado (via SQL DuckDB):")
display(media_tempo_resposta_df)

Média do Tempo de Resposta por Segmento de Mercado (via SQL DuckDB):


,Segmento de Mercado,Media Tempo Resposta
0,Comércio Eletrônico,9.905088
1,Farmácias,9.515826
2,"Material de Construção, Acabamento e Ferramentas",9.189542
3,Aluguel de Carros,9.030000
4,"Fabricantes - Eletroeletrônicos, Produtos de ...",8.884206
5,"Viagens, Turismo e Hospedagem",8.650108
6,Provedores de Conteúdo e Outros Serviços na In...,8.644065
7,Varejo,8.629782
8,"Perfumaria, Cosméticos e Higiene Pessoal",8.552555
9,"Montadoras, Concessionárias e Prestadores de S...",8.466321


In [ ]:
# Define o caminho para salvar o arquivo Parquet
output_parquet_avg_tempo_resposta_path = '/content/drive/MyDrive/Colab Notebooks/2026/arquivos/media_tempo_resposta.parquet'

# Salva o DataFrame em um arquivo Parquet
media_tempo_resposta_df.to_parquet(output_parquet_avg_tempo_resposta_path, index=False)

print(f"DataFrame 'media_tempo_resposta_df' salvo com sucesso em: {output_parquet_avg_tempo_resposta_path}")

DataFrame 'media_tempo_resposta_df' salvo com sucesso em: /content/drive/MyDrive/Colab Notebooks/2026/arquivos/media_tempo_resposta.parquet


Recusadas e posteriormente improcedentes

In [ ]:
sql_query_distinct_recusa_analise = """
SELECT
    "Segmento de Mercado",
    "Análise da Recusa",
    COUNT(*) AS "Total por Análise da Recusa"
FROM
    base_completa_2025_12
WHERE
    TRY_CAST("Data Recusa" AS DATE) IS NOT NULL
GROUP BY
    "Segmento de Mercado",
    "Análise da Recusa"
ORDER BY
    "Segmento de Mercado",
    "Total por Análise da Recusa" DESC
"""

distinct_recusa_analise_df = con.execute(sql_query_distinct_recusa_analise).fetchdf()

print("Total para cada item distinto de 'Análise da Recusa' por 'Segmento de Mercado' (Data Recusa não vazia):")
display(distinct_recusa_analise_df)



Total para cada item distinto de 'Análise da Recusa' por 'Segmento de Mercado' (Data Recusa não vazia):


,Segmento de Mercado,Análise da Recusa,Total por Análise da Recusa
0,Administradoras de Consórcios,Procedente,68
1,Administradoras de Consórcios,Encerrada,23
2,Administradoras de Consórcios,Improcedente,3
3,Agua e Saneamento,Procedente,395
4,Agua e Saneamento,Encerrada,48
...,...,...,...
112,"Vestuário, Calçados e Acessórios",Encerrada,22
113,"Vestuário, Calçados e Acessórios",Improcedente,8
114,"Viagens, Turismo e Hospedagem",Procedente,230
115,"Viagens, Turismo e Hospedagem",Encerrada,61


In [ ]:
# Define o caminho para salvar o arquivo Parquet
output_parquet_distinct_recusa_analise_path = '/content/drive/MyDrive/Colab Notebooks/2026/arquivos/recusa_analise_segmento.parquet'

# Salva o DataFrame em um arquivo Parquet
distinct_recusa_analise_df.to_parquet(output_parquet_distinct_recusa_analise_path, index=False)

print(f"DataFrame 'distinct_recusa_analise_df' salvo com sucesso em: {output_parquet_distinct_recusa_analise_path}")

DataFrame 'distinct_recusa_analise_df' salvo com sucesso em: /content/drive/MyDrive/Colab Notebooks/2026/arquivos/recusa_analise_segmento.parquet
